# Matrices TF, IDF y TF-IDF

Cada archivo `.txt` del corpus es un documento. Las filas de las matrices representan documentos y las columnas representan terminos. TF es el conteo de apariciones del termino en el documento; IDF se calcula como $\ln(N/df(t)) + 1$, donde $N$ es el numero de documentos y $df(t)$ es el numero de documentos que contienen el termino. Finalmente, $TF\text{-}IDF(t,d) = TF(t,d)\times IDF(t)$.

Las matrices completas se conservan en memoria como matrices dispersas; las tablas muestran solo una muestra para facilitar su lectura.

In [34]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

## 1. Conectar y leer el corpus

Se buscan recursivamente todos los archivos `.txt` en la ruta indicada. Se usa UTF-8 con deteccion de marca BOM; si un archivo no puede leerse, se informa el error en lugar de modificar su contenido silenciosamente.

In [35]:
ruta_corpus = Path(r"C:\Users\bravo\PycharmProjects\DataRI\Tarea2\txt-files.tar\cache\Epub2")
if not ruta_corpus.is_dir():
    raise FileNotFoundError(f"No se encontró la carpeta del corpus: {ruta_corpus}")

archivos = sorted(ruta_corpus.rglob("*.txt"))
if not archivos:
    raise ValueError(f"No se encontraron archivos .txt en {ruta_corpus}")

documentos = [ruta.relative_to(ruta_corpus).as_posix() for ruta in archivos]
N = len(archivos)
print(f"Documentos encontrados: {N}")
display(pd.DataFrame({"documento": documentos}).head(10))

Documentos encontrados: 96


,documento
0,1/pg1.txt
1,10/pg10.txt
2,100/pg100.txt
3,11/pg11.txt
4,12/pg12.txt
5,13/pg13.txt
6,14/pg14.txt
7,15/pg15.txt
8,16/pg16.txt
9,17/pg17.txt


## 2. Matriz de frecuencia de terminos (TF)

Se convierte el texto a minusculas y se extraen secuencias de letras y numeros, incluidos terminos de un solo caracter. No se eliminan palabras frecuentes ni se aplica stemming. Cada valor es un conteo absoluto, sin normalizacion.

In [36]:
vectorizador = CountVectorizer(
    input="filename", encoding="utf-8-sig", decode_error="strict",
    lowercase=True, token_pattern=r"(?u)\b\w+\b", dtype=np.int64,
)
matriz_tf = vectorizador.fit_transform([str(ruta) for ruta in archivos]).tocsr()
terminos = vectorizador.get_feature_names_out()
print(f"Matriz TF: {matriz_tf.shape[0]} documentos x {matriz_tf.shape[1]} terminos")
print(f"Valores distintos de cero: {matriz_tf.nnz:,}")

# La muestra es pequena; matriz_tf contiene todos los documentos y terminos.
filas_muestra, columnas_muestra = 5, 15
display(pd.DataFrame(
    matriz_tf[:filas_muestra, :columnas_muestra].toarray(),
    index=documentos[:filas_muestra], columns=terminos[:columnas_muestra],
).rename_axis(index="documento", columns="termino"))

Matriz TF: 96 documentos x 133429 terminos
Valores distintos de cero: 689,593


termino,0,00,000,0000,00000,000000,0000000,0000000000,000000000000000000,00000000000000000000000000000000000000000000000000,0000010,0000101,0000110,000015,000017
documento,,,,,,,,,,,,,,,
1/pg1.txt,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0
10/pg10.txt,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
100/pg100.txt,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
11/pg11.txt,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0
12/pg12.txt,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


## 3. Frecuencia inversa de documentos (IDF)

Se cuenta cada documento una sola vez por termino. Se usa logaritmo natural y la formula solicitada, sin suavizar $N$ ni $df(t)$; todo termino del vocabulario aparece al menos en un documento.

In [37]:
df = np.asarray((matriz_tf > 0).sum(axis=0)).ravel()
idf = np.log(N / df) + 1
tabla_idf = pd.DataFrame({"termino": terminos, "df": df, "idf": idf}).set_index("termino")
print(f"Terminos con IDF calculado: {len(tabla_idf):,}")
display(tabla_idf.head(15))

Terminos con IDF calculado: 133,429


,df,idf
termino,,
0,23,2.428854
00,17,2.731135
000,96,1.000000
0000,7,3.618438
00000,1,5.564348
000000,1,5.564348
0000000,1,5.564348
0000000000,1,5.564348
000000000000000000,1,5.564348


## 4. Matriz TF-IDF

Se multiplica cada columna de la matriz TF por el IDF de su termino. No se aplica normalizacion adicional, de modo que los valores corresponden directamente a la formula indicada.

In [38]:
matriz_tfidf = matriz_tf.multiply(idf).tocsr()
print(f"Matriz TF-IDF: {matriz_tfidf.shape[0]} documentos x {matriz_tfidf.shape[1]} terminos")
display(pd.DataFrame(
    matriz_tfidf[:filas_muestra, :columnas_muestra].toarray(),
    index=documentos[:filas_muestra], columns=terminos[:columnas_muestra],
).rename_axis(index="documento", columns="termino"))

Matriz TF-IDF: 96 documentos x 133429 terminos


termino,0,00,000,0000,00000,000000,0000000,0000000000,000000000000000000,00000000000000000000000000000000000000000000000000,0000010,0000101,0000110,000015,000017
documento,,,,,,,,,,,,,,,
1/pg1.txt,0.000000,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10/pg10.txt,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100/pg100.txt,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11/pg11.txt,2.428854,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12/pg12.txt,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Consultar un termino y comprobar el calculo

Cambie `termino_consulta` para examinar el termino en todos los documentos. La tabla muestra tambien los ceros; los valores completos estan disponibles en `matriz_tf`, `tabla_idf` y `matriz_tfidf`.

In [39]:
termino_consulta = "the"
if termino_consulta in vectorizador.vocabulary_:
    columna = vectorizador.vocabulary_[termino_consulta]
    resultado = pd.DataFrame({
        "documento": documentos,
        "tf": matriz_tf[:, columna].toarray().ravel(),
        "idf": idf[columna],
        "tf_idf": matriz_tfidf[:, columna].toarray().ravel(),
    }).set_index("documento")
    print(f"Termino: {termino_consulta!r}; df = {df[columna]}; IDF = {idf[columna]:.6f}")
    display(resultado.head(10))
else:
    print(f"El termino {termino_consulta!r} no aparece en el vocabulario.")

assert matriz_tf.shape == matriz_tfidf.shape
assert np.all(df > 0) and np.all(df <= N)
assert np.allclose(idf, np.log(N / df) + 1)
print("Comprobacion de dimensiones y formula: correcta.")

Termino: 'the'; df = 96; IDF = 1.000000


,tf,idf,tf_idf
documento,,,
1/pg1.txt,306,1.0,306.0
10/pg10.txt,64309,1.0,64309.0
100/pg100.txt,30526,1.0,30526.0
11/pg11.txt,1839,1.0,1839.0
12/pg12.txt,1795,1.0,1795.0
13/pg13.txt,567,1.0,567.0
14/pg14.txt,5602,1.0,5602.0
15/pg15.txt,14718,1.0,14718.0
16/pg16.txt,2529,1.0,2529.0


Comprobacion de dimensiones y formula: correcta.
